# 10주차 실습 — 안전한 데이터 수정과 트랜잭션

- **데이터**: `studies` 에서 5행만 떼어 만든 연습용 표 `studies_copy`. 원본은 그대로입니다.
- **오늘 배우는 것**: 추가(INSERT) · 수정(UPDATE) · 삭제(DELETE), 그리고 되돌리기(ROLLBACK)
- 표가 꼬이면 `처음상태로()` 를 실행하면 처음 5행으로 돌아갑니다.

## 0. 시작하기 전에

- Colab 에서는 먼저 **`File → Save a copy in Drive`** 를 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` · `-- S1` 같은 문항 이름 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.

In [ ]:
#@title [1번] ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
_pd.set_option("display.max_columns", None)     # SELECT * 로 열 이름을 볼 때 모든 열을 보여 줌
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

ANSWERS = _load_answers("eNq9lV1PGkEUhv/KCTdCQgjflCa9sEqUNGKgGC+axvCxtCYUGqDthTFZKxqqNIWGlcUCwYQqWi+o0ooJvelP6eXO2f/QMysoYNukaZUbZt85c3bmfc6ZXdH5Lbq7sKJLLz+hf13IJghRISZYrWGbzgi65G/0VPJVmnQnDSPJOB8+0iUimaXlqLbqpZAKxeNL6Uwo8yLNFSGRSsbjz4RERveYHp8u04DSKidt5bQHakXCWhfUYhXYQQtQyrK6jOUC+KaCbreZ/yygSlVQd95grQPstIPlY5RopXTItnNsuwl6J00aTKC032NNBKWTVeUupcoBljuAuSpmqywvsr2qEZSuiPUC2yoB1osUzUptlDv0eiMFfupPKWfdi3gTUH7cprdLDWBSXlt3ImMtj+tr6nqVQvQGwA2RfRXVbBtwu0lbUSUZs13cLZk0R1JRISVE6cyxUDwtcOV5ZjmZCMUH0ippfuswC3PE5Xa5LPaIPeoeZjGu91lYhlnQdtWyjDn5F3bz/duAfMSdzoiHJlic9QQ8wJ2f4K7d425OADtq4U4Rqz06KWBjDbe+DBZQFHd7FLhxAGFi0RucnQ5MLvomQM/NPO6SucZx1w1/Y5FttFxd4Zjb6rLecZlHy3VU71vk+N/lSk5RWcHlMcFG5NlBHoKewJzXNxn0TJMXIrCCfN1sB8Vyb1mxAKxdYmIeyznu8mGnD4J97AE7b7Bqi+23QTlZx/rGYPVDT5CD0opxPccdN17h46rfClihRmhXOArcayufiVV2DOCtlbZ9pLTD0ZgtGg47XVHXSGmP6X1u1hvgBlY6iX7B98A3T9ycZjN8P4OAZyqw4A16fTNgMRs4SPa6RdfRGDnKoFE9z/6BWKM5gIsHIu52xvhdXTSXfUDkNvn1hqW1W8fjGMZjj9nc5og7Yo9FnMN4xvWbaitu7sU5tSbRX/UXMRogG8X1Q9w39G8o9i7PkVXEcWqUvyHpp+bn5rxBg2Y0fWhostaD+54Zr4/fieqHLNZk+sKIysk3rV3Km+ytTFNabvJWQ3a0wRrnQzSvc/I7AJsilci/81n9Cd8pg9I=")   # 자동 확인용 (정답 SQL 은 없음)

In [ ]:
#@title [2번] ▶ 수정 연습 준비 — 실행만 하세요 (Shift+Enter) { display-mode: "form" }
class _한연결:
    """%%sql 셀이 BEGIN 을 다음 셀의 COMMIT · ROLLBACK 까지 이어 가게 합니다."""
    def __init__(self, con): self.con = con
    def cursor(self): return self
    def commit(self): pass
    def close(self): pass
    def __getattr__(self, name): return getattr(self.con, name)

if "_rw" not in globals():
    _rw = _한연결(conn)
%sql _rw --alias rw

def 처음상태로():
    """studies_copy 를 처음 5행으로 되돌립니다."""
    try:
        _rw.execute("ROLLBACK")
    except Exception:
        pass
    _rw.execute("""CREATE OR REPLACE TEMP TABLE studies_copy AS
SELECT nct_id, overall_status, enrollment
FROM studies
ORDER BY nct_id LIMIT 5;""")
    print("studies_copy 를 처음 5행으로 되돌렸습니다.")

print("준비 완료")

## 1. INSERT · UPDATE · DELETE

### 연습용 표 만들기

In [ ]:
%%sql
-- [3번] 연습용 표 → 5행
CREATE OR REPLACE TEMP TABLE studies_copy AS
SELECT nct_id, overall_status, enrollment
FROM studies
ORDER BY nct_id LIMIT 5;

SELECT * FROM studies_copy ORDER BY nct_id;

### INSERT — 새 행 추가 (5행 → 6행)

**모양** `INSERT INTO 표 이름 (열, …) VALUES (값, …)` — 열 목록과 값은 같은 순서

In [ ]:
처음상태로()   # [4번]

In [ ]:
%%sql
-- [5번] INSERT → 6행
INSERT INTO studies_copy
  (nct_id, overall_status, enrollment)
VALUES ('NCT99999999', 'RECRUITING', 50);

SELECT * FROM studies_copy ORDER BY nct_id;

### UPDATE — 값 바꾸기 (WHERE 에 맞는 1행만)

**모양** `UPDATE 표 이름 SET 열 = 새 값 WHERE 조건` — SET 은 꼭 씀, WHERE 가 없으면 모든 행

In [ ]:
처음상태로()   # [6번]

In [ ]:
%%sql
-- [7번] UPDATE → NCT00623389 만 COMPLETED
UPDATE studies_copy
SET overall_status = 'COMPLETED'
WHERE nct_id = 'NCT00623389';

SELECT * FROM studies_copy ORDER BY nct_id;

### DELETE — 행 지우기 (WITHDRAWN 3행 → 2행 남음)

**모양** `DELETE FROM 표 이름 WHERE 조건` — 행을 통째로 지우므로 SET 이 없음

In [ ]:
처음상태로()   # [8번]

In [ ]:
%%sql
-- [9번] DELETE → 2행
DELETE FROM studies_copy
WHERE overall_status = 'WITHDRAWN';

SELECT * FROM studies_copy ORDER BY nct_id;

### WHERE 를 빠뜨리면 — 모든 행이 바뀝니다

In [ ]:
처음상태로()   # [10번]

In [ ]:
%%sql
-- [11번] WHERE 없음 → 5행 모두 TERMINATED (아래 ②와 WHERE 만 다름)
UPDATE studies_copy
SET overall_status = 'TERMINATED';
-- WHERE 가 없습니다

SELECT * FROM studies_copy ORDER BY nct_id;

### 그래서 — 같은 WHERE 로 먼저 세고, 바꿉니다

In [ ]:
처음상태로()   # [12번]

In [ ]:
%%sql
-- [13번] ① 먼저 세기 → 3
SELECT COUNT(*) FROM studies_copy
WHERE overall_status = 'WITHDRAWN';

In [ ]:
%%sql
-- [14번] ② 같은 WHERE 로 바꾸기 → 3행만 TERMINATED
UPDATE studies_copy
SET overall_status = 'TERMINATED'
WHERE overall_status = 'WITHDRAWN';

SELECT * FROM studies_copy ORDER BY nct_id;

## 2. 트랜잭션

여러 변경을 한 덩어리로 묶는 것. 예) 입원 수속 = ① 입원 기록 ② 병상 ‘사용 중’ ③ 수납 내역 — 하나라도 실패하면 전부 없던 일로 되돌립니다.

- 병원 프로그램: EMR 의 [저장] = 묶인 변경을 함께 확정, [취소] · 오류 = 모두 없던 일로. 프로그램이 대신 묶어 줍니다.
- DBMS 가 지켜 주는 4가지(ACID): 전부 아니면 전무 · 규칙 지키기 · 서로 섞이지 않기 · 확정하면 남기

## 3. COMMIT과 ROLLBACK

`BEGIN;` 뒤에 바꾼 것은 **COMMIT** 하면 확정, **ROLLBACK** 하면 취소됩니다. 확정한 뒤에는 되돌릴 수 없습니다.

### ROLLBACK 해 보기 — 실수를 되돌리기

In [ ]:
처음상태로()   # [15번]

In [ ]:
%%sql
-- [16번] ① 실수: 5행 모두 COMPLETED (아직 확정 전)
BEGIN;

UPDATE studies_copy
SET overall_status = 'COMPLETED';   -- WHERE 없음

SELECT * FROM studies_copy ORDER BY nct_id;

In [ ]:
%%sql
-- [17번] ② ROLLBACK → 처음 5행대로
ROLLBACK;

SELECT * FROM studies_copy ORDER BY nct_id;

### COMMIT 해 보기 — 확정하면 끝

In [ ]:
처음상태로()   # [18번]

In [ ]:
%%sql
-- [19번] COMMIT → NCT00623389 가 COMPLETED 로 확정
BEGIN;

UPDATE studies_copy SET overall_status = 'COMPLETED'
WHERE nct_id = 'NCT00623389';

COMMIT;   -- 확정

SELECT * FROM studies_copy ORDER BY nct_id;

In [ ]:
%%sql
-- [20번] [오류 예시] 확정한 뒤라 되돌릴 것이 없음 (no transaction is active)
ROLLBACK;

In [ ]:
%%sql
-- [21번] 확정된 그대로 (NCT00623389 = COMPLETED)
SELECT * FROM studies_copy ORDER BY nct_id;

### 한 덩어리로 되돌리기 — 두 변경을 함께 취소

In [ ]:
처음상태로()   # [22번]

In [ ]:
%%sql
-- [23번] ①② 뒤 → 2행만 남음 (아직 확정 전)
BEGIN;

UPDATE studies_copy SET overall_status = 'COMPLETED'
WHERE nct_id = 'NCT00623389';   -- ① 바꾸기

DELETE FROM studies_copy
WHERE overall_status = 'WITHDRAWN';   -- ② 지우기

SELECT * FROM studies_copy ORDER BY nct_id;

In [ ]:
%%sql
-- [24번] ROLLBACK 한 번 → ①② 함께 취소, 처음 5행
ROLLBACK;

SELECT * FROM studies_copy ORDER BY nct_id;

### BEGIN 없이 바꾸면 — 바로 확정

In [ ]:
처음상태로()   # [25번]

In [ ]:
%%sql
-- [26번] BEGIN 없이 UPDATE → 바로 확정
UPDATE studies_copy
SET overall_status = 'COMPLETED';   -- BEGIN 없음

SELECT * FROM studies_copy ORDER BY nct_id;

In [ ]:
%%sql
-- [27번] [오류 예시] no transaction is active — 되돌릴 트랜잭션이 없음
ROLLBACK;

In [ ]:
%%sql
-- [28번] 그대로 모두 COMPLETED → 되돌리려면 먼저 BEGIN
SELECT * FROM studies_copy ORDER BY nct_id;

## 직접 해보기 — 5문항

문항마다 `처음상태로()` 셀부터 실행하고, `___` 를 채워 실행하면 ✅ / ❌ 가 나옵니다. 열 이름은 아래 표에서 봅니다.

In [ ]:
%%sql
-- [29번] studies_copy 의 열: nct_id · overall_status · enrollment
SELECT *
FROM (DESCRIBE studies_copy);

**Q1.** **추가** — 새 시험 `'NCT99000001'`, `'RECRUITING'`, `40` 을 넣으세요. → 6행이 되면 정답

In [ ]:
처음상태로()   # [30번]

In [ ]:
%%sql
-- [31번] Q1
INSERT INTO studies_copy (nct_id, overall_status, enrollment)
VALUES (___, ___, ___);

-- 결과 확인 (그대로 두세요)
SELECT * FROM studies_copy ORDER BY nct_id;

**Q2.** **세기** — 상태(`overall_status`)가 `'WITHDRAWN'` 인 시험은 몇 건? → 3

In [ ]:
처음상태로()   # [32번]

In [ ]:
%%sql
-- [33번] Q2
SELECT COUNT(*) AS 시험수
FROM studies_copy
WHERE ___;

**Q3.** **바꾸기** — `'WITHDRAWN'` 인 시험을 `'TERMINATED'` 로 바꾸세요. → 3행만 바뀌면 정답

In [ ]:
처음상태로()   # [34번]

In [ ]:
%%sql
-- [35번] Q3
UPDATE studies_copy
SET overall_status = ___
WHERE ___;

-- 결과 확인 (그대로 두세요)
SELECT * FROM studies_copy ORDER BY nct_id;

**Q4.** **지우기** — 등록 인원(`enrollment`)이 `0` 인 시험을 지우세요. → 2행이 남으면 정답

In [ ]:
처음상태로()   # [36번]

In [ ]:
%%sql
-- [37번] Q4
DELETE FROM studies_copy
WHERE ___;

-- 결과 확인 (그대로 두세요)
SELECT * FROM studies_copy ORDER BY nct_id;

**Q5.** **되돌리기** — 준비 셀이 실수를 했습니다 (BEGIN 뒤 WHERE 없이 모두 COMPLETED). `___` 에 되돌리는 명령을 쓰세요. → 처음 5행이면 정답

In [ ]:
# [38번]
# Q5 준비: BEGIN 후 WHERE 없는 UPDATE
처음상태로()
%sql BEGIN;
%sql UPDATE studies_copy SET overall_status = 'COMPLETED';
%sql SELECT * FROM studies_copy;

In [ ]:
%%sql
-- [39번] Q5
___;

-- 결과 확인 (그대로 두세요)
SELECT * FROM studies_copy ORDER BY nct_id;

## 오늘 배운 것

- 추가 · 수정 · 삭제 명령이 있다. WHERE로 대상을 정한다
- 여러 변경을 묶고, 확정하거나 확정 전에 되돌릴 수 있다
- 연습은 사본에서 하고, 원본은 그대로 둔다

다음 시간: 11강. CDW 구조와 분석 업무 흐름 — 병원에서 데이터를 받는 과정

In [ ]:
진행확인()   # [40번]